# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [18]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

RANDOM_STATE = 0

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
#your code here
spaceship.shape

(8693, 14)

**Check for data types**

In [4]:
#your code here
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [5]:
#your code here
spaceship_missing = spaceship[spaceship.isna().any(axis=1)]
spaceship_missing.shape

(2087, 14)

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
#your code here
spaceship = spaceship.dropna()

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [7]:
#your code here
spaceship['Cabin_section'] = spaceship['Cabin'].astype(str).str[0]
spaceship = spaceship.drop(columns='Cabin')
spaceship

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Cabin_section
0,0001_01,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B
1,0002_01,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F
2,0003_01,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A
3,0003_02,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A
4,0004_01,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False,A
8689,9278_01,Earth,True,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False,G
8690,9279_01,Earth,False,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True,G
8691,9280_01,Europa,False,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False,E


- Drop PassengerId and Name

In [8]:
#Drop columns PassengerID and Name
spaceship_df = spaceship.drop(['PassengerId', 'Name'], axis=1)

In [102]:
display(spaceship_df)

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Cabin_section
0,Europa,0,TRAPPIST-1e,39.0,0,0.0,0.0,0.0,0.0,0.0,False,B
1,Earth,0,TRAPPIST-1e,24.0,0,109.0,9.0,25.0,549.0,44.0,True,F
2,Europa,0,TRAPPIST-1e,58.0,1,43.0,3576.0,0.0,6715.0,49.0,False,A
3,Europa,0,TRAPPIST-1e,33.0,0,0.0,1283.0,371.0,3329.0,193.0,False,A
4,Earth,0,TRAPPIST-1e,16.0,0,303.0,70.0,151.0,565.0,2.0,True,F
...,...,...,...,...,...,...,...,...,...,...,...,...
8688,Europa,0,55 Cancri e,41.0,1,0.0,6819.0,0.0,1643.0,74.0,False,A
8689,Earth,1,PSO J318.5-22,18.0,0,0.0,0.0,0.0,0.0,0.0,False,G
8690,Earth,0,TRAPPIST-1e,26.0,0,0.0,0.0,1872.0,1.0,0.0,True,G
8691,Europa,0,55 Cancri e,32.0,0,0.0,1049.0,0.0,353.0,3235.0,False,E


**Perform Train Test Split**

In [9]:
#Perform Train Test Split on spaceship_df
X_spaceship = spaceship_df.drop(columns='Transported')
y_spaceship = spaceship_df['Transported']
X_train_spaceship, X_test_spaceship, y_train_spaceship, y_test_spaceship = train_test_split(
    X_spaceship, y_spaceship, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y_spaceship
)

### For non-numerical columns, do dummies.

In [10]:
#your code here
spaceship_df["CryoSleep"] = spaceship_df["CryoSleep"].astype(int)
spaceship_df["VIP"] = spaceship_df["VIP"].astype(int)
spaceship_df.dtypes
print(spaceship_df["Destination"].unique())
print(spaceship_df["HomePlanet"].unique())
display(spaceship_df.describe())

['TRAPPIST-1e' 'PSO J318.5-22' '55 Cancri e']
['Europa' 'Earth' 'Mars']


,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,6606.000000,6606.000000,6606.000000,6606.000000,6606.000000,6606.000000,6606.000000,6606.000000
mean,0.353012,28.894036,0.024523,222.991674,478.958523,178.356494,313.161520,303.780048
std,0.477943,14.533429,0.154678,644.987936,1678.592291,576.328407,1144.016291,1127.142166
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,38.000000,0.000000,49.000000,82.750000,30.000000,65.000000,52.000000
max,1.000000,79.000000,1.000000,9920.000000,29813.000000,12253.000000,22408.000000,20336.000000


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [11]:
#your code here
#Define nominal features for HomePlanet, Destination, and Cabin_section 
nominal_features = ['HomePlanet', 'Destination', 'Cabin_section', 'CryoSleep', 'VIP']
numerical_features = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

nominal_pipeline = Pipeline([('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
#nominal_pipeline.fit(X_train_spaceship[nominal_features])
#nominal_pipeline.transform(X_train_spaceship[nominal_features])
numerical_pipeline =  Pipeline([('minmax', MinMaxScaler())])

processing = ColumnTransformer([
    ('nominal', nominal_pipeline, nominal_features),
    ('numerical', numerical_pipeline, numerical_features)
])

full_pipeline = Pipeline([('preprocess', processing), ('model', KNeighborsClassifier(n_neighbors=5))])

full_pipeline.fit(X_train_spaceship, y_train_spaceship)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('nominal', ...), ('numerical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


- Evaluate your model's performance. Comment it

In [12]:
print('Training accuracy:', round(full_pipeline.score(X_train_spaceship, y_train_spaceship), 3))
print('Test accuracy:', round(full_pipeline.score(X_test_spaceship, y_test_spaceship), 3))

Training accuracy: 0.828
Test accuracy: 0.751


In [15]:
# predictions for full pipeline
y_pred_train_knn = full_pipeline.predict(X_train_spaceship)
y_pred_test_knn  = full_pipeline.predict(X_test_spaceship)

comparison_spaceship = pd.DataFrame({
    "actual_transport_status": y_test_spaceship.map({False: "Stayed", True: "Transported"}),
    "predicted_transport_status": pd.Series(y_pred_test_knn, index=y_test_spaceship.index).map({False: "Stayed", True: "Transported"}),
})

display(comparison_spaceship.head(10))

,actual_transport_status,predicted_transport_status
2236,Stayed,Stayed
2523,Transported,Transported
2204,Transported,Transported
8069,Stayed,Stayed
404,Stayed,Stayed
7335,Stayed,Stayed
1941,Stayed,Stayed
8259,Transported,Stayed
5737,Stayed,Stayed
1068,Stayed,Stayed


In [19]:
# Determine the accuracy, precision, and recall
classification_metrics = pd.DataFrame({
    "Training": {
        "Accuracy": accuracy_score(y_train_spaceship, y_pred_train_knn),
        "Precision (Transported)": precision_score(y_train_spaceship, y_pred_train_knn, pos_label=1),  # CHANGED label
        "Recall (Transported)": recall_score(y_train_spaceship, y_pred_train_knn, pos_label=1),        # CHANGED label
    },
    "Test": {
        "Accuracy": accuracy_score(y_test_spaceship, y_pred_test_knn),
        "Precision (Transported)": precision_score(y_test_spaceship, y_pred_test_knn, pos_label=1),    # CHANGED label
        "Recall (Transported)": recall_score(y_test_spaceship, y_pred_test_knn, pos_label=1),          # CHANGED label
    },
})
display(classification_metrics.round(3))

,Training,Test
Accuracy,0.828,0.751
Precision (Transported),0.834,0.756
Recall (Transported),0.822,0.748
